In [37]:
import SimpleITK as sitk 
import os 
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact, IntSlider
from tqdm import tqdm
import matplotlib.colors as mcolors

In [38]:
def run_staple(file_paths: list[str], foreground_value: int | list[int] = 1):
    segmentations = []
    if isinstance(foreground_value, int):
        foreground_value = [foreground_value] * len(file_paths)
    for label, path in zip(foreground_value, file_paths):
        image = sitk.ReadImage(path)
        binary_img = sitk.Cast(image == label, sitk.sitkUInt8)
        segmentations.append(binary_img)

    staple_filter = sitk.STAPLEImageFilter()
    staple_filter.SetForegroundValue(1)

    probability_map = staple_filter.Execute(segmentations)

    consensus_segmentation = sitk.Cast(probability_map > 0.5, sitk.sitkUInt8)

    sensitivities = staple_filter.GetSensitivity()
    specificities = staple_filter.GetSpecificity()
    iterations = staple_filter.GetElapsedIterations()

    print(f"STAPLE converged after {iterations} iterations.")

    pbar = tqdm(enumerate(zip(sensitivities, specificities)), 
                total=len(sensitivities), 
                desc="Calculating Observer Performance")
    
    for i, (sens, spec) in pbar:
        tqdm.write(f"Observer {i}: Sensitivity = {sens:.4f}, Specificity = {spec:.4f}")

    return consensus_segmentation

In [43]:
def visualize_staple_comparison(original_img, input_masks, consensus_mask, ground_truth=None, titles=None, foreground_value: int | list[int] = 1):
    """
    Visualizes original image, expert masks, STAPLE consensus, and Ground Truth.
    Adds a final panel comparing STAPLE vs Ground Truth with improved color contrast.
    """
    img_arr = sitk.GetArrayFromImage(original_img)
    staple_arr = sitk.GetArrayFromImage(consensus_mask)
    
    processed_masks = []
    for i, m in enumerate(input_masks):
        m_arr = sitk.GetArrayFromImage(m)
        val = foreground_value if isinstance(foreground_value, int) else foreground_value[i]
        processed_masks.append(m_arr == val)
    
    gt_arr = None
    if ground_truth is not None:
        gt_arr = sitk.GetArrayFromImage(ground_truth)
        gt_arr = (gt_arr > 0).astype(np.uint8) 

    display_masks = list(processed_masks) + [staple_arr]
    display_titles = (titles if titles else [f"Expert {i+1}" for i in range(len(processed_masks))]) + ["STAPLE Consensus"]
    
    if gt_arr is not None:
        display_masks.append(gt_arr)
        display_titles.append("Ground Truth")

    combined_vol = sum([m.astype(float) for m in processed_masks]) + staple_arr
    if gt_arr is not None: combined_vol += gt_arr
    
    has_content = np.any(combined_vol > 0, axis=(1, 2))
    slice_indices = np.where(has_content)[0]
    
    if len(slice_indices) == 0:
        print("No foreground found in any masks.")
        return

    min_slice, max_slice = int(slice_indices.min()), int(slice_indices.max())

    def plot_slice(z):
        total_plots = len(display_masks) + (1 if gt_arr is not None else 0)
        cols = 3
        rows = (total_plots + cols - 1) // cols
        
        _, axes = plt.subplots(rows, cols, figsize=(18, 5 * rows))
        axes = axes.flatten()

        for i in range(len(display_masks)):
            axes[i].imshow(img_arr[z, :, :], cmap='gray')
            mask_slice = display_masks[i][z, :, :]
            
            if np.any(mask_slice):
                masked_data = np.ma.masked_where(mask_slice == 0, mask_slice)
                
                if display_titles[i] == "Ground Truth":
                    color = 'lime'
                elif display_titles[i] == "STAPLE Consensus":
                    color = 'springgreen' 
                else:
                    color = 'deepskyblue'

                
                cmap = mcolors.ListedColormap([color])
                axes[i].imshow(masked_data, cmap=cmap, alpha=0.5, vmin=0, vmax=1)
            
            axes[i].set_title(f"{display_titles[i]} (Slice {z})")
            axes[i].axis('off')

        if gt_arr is not None:
            idx = len(display_masks)
            axes[idx].imshow(img_arr[z, :, :], cmap='gray')
            
            g_slice = np.ma.masked_where(gt_arr[z, :, :] == 0, gt_arr[z, :, :])
            gt_cmap = mcolors.ListedColormap(['lime'])
            axes[idx].imshow(g_slice, cmap=gt_cmap, alpha=0.6, vmin=0, vmax=1)

            
            s_slice = np.ma.masked_where(staple_arr[z, :, :] == 0, staple_arr[z, :, :])
            staple_cmap = mcolors.ListedColormap(['white'])
            axes[idx].imshow(s_slice, cmap=staple_cmap, alpha=0.4, vmin=0, vmax=1)
            
            axes[idx].set_title(f"Overlay: GT (Cyan) vs STAPLE (White)")
            axes[idx].axis('off')
            last_idx = idx
        else:
            last_idx = len(display_masks) - 1

        
        for j in range(last_idx + 1, len(axes)):
            axes[j].axis('off')
            
        plt.tight_layout(h_pad=3.0)
        plt.show()

    interact(plot_slice, z=IntSlider(
        min=min_slice, max=max_slice, step=1, 
        value=(min_slice + max_slice) // 2,
        description='Slice:'
    ))

In [45]:
segmentation_paths = [
    "/home/anastasiia/MasterThesis/PancreasMRISegmentation/results1/mri_segmentator/panther/10000_0001.nii.gz", 
    "/home/anastasiia/MasterThesis/PancreasMRISegmentation/results1/mri_segmenter/panther/10000_0001.nii.gz",
    "/home/anastasiia/MasterThesis/PancreasMRISegmentation/results1/total_segmentator/panther/10000_0001.nii.gz",
    "/home/anastasiia/MasterThesis/PancreasMRISegmentation/results1/pansegnet_t1/panther/10000_0001.nii.gz", 
    "/home/anastasiia/MasterThesis/PancreasMRISegmentation/results1/umamba_bot/panther/10000_0001.nii.gz", 
    "/home/anastasiia/MasterThesis/PancreasMRISegmentation/results1/umamba_enc/panther/10000_0001.nii.gz"
]
original_image_path = "/home/anastasiia/MasterThesis/external_data/Imaging/Panther/nnUNet_raw/Dataset090_PantherTask1/imagesTr/10000_0001_0000.nii.gz"
ground_truth_path = "/home/anastasiia/MasterThesis/external_data/Imaging/Panther/nnUNet_raw/Dataset090_PantherTask1/labelsTr/10000_0001.nii.gz"

In [29]:
conseunsus_mask = run_staple(segmentation_paths, foreground_value=[7, 11, 7])

STAPLE converged after 10 iterations.


Calculating Observer Performance: 100%|██████████| 3/3 [00:00<00:00, 1030.12it/s]

Observer 0: Sensitivity = 0.8877, Specificity = 1.0000
Observer 1: Sensitivity = 0.9940, Specificity = 0.9993
Observer 2: Sensitivity = 0.9286, Specificity = 0.9999


In [47]:
visualize_staple_comparison(
    sitk.ReadImage(original_image_path), 
    [sitk.ReadImage(p) for p in segmentation_paths], 
    conseunsus_mask, 
    ground_truth=sitk.ReadImage(ground_truth_path),
    titles=["MRI Segmentator", "MRI Segmenter", "Total Segmentator", "PaNSegNet T1", "UMaMBa Bot", "UMaMBa Enc"],
    foreground_value=[7, 11, 7, 1, 4, 4]
)

interactive(children=(IntSlider(value=30, description='Slice:', max=44, min=16), Output()), _dom_classes=('wid…